# dict_comprehensions
Dictionary comprehensions are for building keyed mappings compactly from iterables. They are powerful in indexing, grouping, projection, and config shaping, but they can silently overwrite keys if you do not think carefully about uniqueness.

## 1. Problem First
Why does this matter in real systems?
- Indexing records by ID is a common data-engineering operation.
- Config and payload transformations often become key-value remaps.
- Duplicate keys can cause silent data loss.

In [ ]:
services = ["api", "worker", "scheduler"]
service_lengths = {service: len(service) for service in services}
print(service_lengths)

## 2. Minimal Concept
Core syntax:
- `{key: value for item in iterable}`
- `{key: value for item in iterable if condition}`
- They build a new dictionary eagerly.

## 3. Mental Model
How Python actually behaves
A dict comprehension iterates, computes a key and value, and inserts them into a new dictionary. If the same key appears more than once, the later value wins. That last-write-wins behavior is convenient sometimes and dangerous other times.

In [ ]:
pairs = [("a", 1), ("b", 2), ("a", 3)]
result = {key: value for key, value in pairs}
print(result)

## 4. Internal Mechanics
Like list comprehensions, dict comprehensions build results eagerly. The key difference is collision behavior: repeated keys overwrite earlier values. The result is insertion-ordered in modern Python, but uniqueness is still by key.

In [ ]:
import dis

def index_by_id(records):
    return {record["id"]: record for record in records}

dis.dis(index_by_id)
print(index_by_id([{"id": 1}, {"id": 2}]))

## 5. Edge Cases
Where it breaks:
- Duplicate keys overwrite earlier entries silently.
- Complex expressions make comprehensions unreadable quickly.
- Missing keys or malformed records can raise exceptions deep inside the expression.
- A normal loop may be better when validation and logging matter.

In [ ]:
records = [{"id": 1, "name": "a"}, {"id": 1, "name": "b"}]
indexed = {record["id"]: record["name"] for record in records}
print(indexed)

## 6. Performance Thinking
- Dict comprehensions are typically O(n) for n input items.
- Memory cost is O(n) for the built mapping.
- They are often much faster than repeated linear searches because the result enables O(1) lookups later.
- The real cost of mistakes is silent key overwrite.

## 7. Real Use Case
A service registry can index instances by hostname or ID for fast lookup during routing and health checks.

In [ ]:
instances = [
    {"id": "a1", "host": "api-1"},
    {"id": "a2", "host": "api-2"}
]
by_id = {instance["id"]: instance["host"] for instance in instances}
print(by_id)

## 8. Anti-Pattern
What beginners do wrong:
- Build a dict comprehension without checking key uniqueness.
- Hide too much validation inside one expression.
- Use it where a loop with logging would be much clearer.

In [ ]:
users = [{"email": "a@example.com"}, {"email": "a@example.com"}]
user_map = {user["email"]: user for user in users}
print(user_map)

## 9. Interview Signals
Questions FAANG asks:
- What happens when a dict comprehension sees duplicate keys?
- When is a dict comprehension better than a normal loop?
- Why is building an index often a performance win?
- What are the readability limits of comprehension-based transformations?

## 10. Exercise (Non-trivial)
Design an indexing step for API records keyed by `id`. Handle duplicates, malformed rows, and optional fields explicitly. Decide whether the result should fail fast on duplicates, keep the first record, or keep the last, and justify that choice.

In [ ]:
def build_index(records):
    return {record["id"]: record for record in records}

# Task:
# 1. Detect duplicate ids.
# 2. Decide how to handle malformed records.
# 3. Explain when a loop is safer than a comprehension.